# 2. Key Vault and Defender for Cloud

This notebook covers two core Azure security services:

1. **Azure Key Vault** — where you store secrets, keys, and certificates.
2. **Microsoft Defender for Cloud** — Azure's CSPM + CWPP dashboard.

Each concept comes with a **bad practice → best practice** comparison plus runnable Python simulations.

## 🔧 Setup (run this once)

Before running the code cells, make sure you've installed dependencies and selected the right kernel:

```bash
cd security-certs/sc-900/03-azure-security-solutions
uv sync
```

Then in **VS Code**: click the kernel picker in the top-right of this notebook and pick the `.venv` interpreter for this folder. If it doesn't show up, reload the window (`Cmd+Shift+P` → *Developer: Reload Window*).

> ℹ️ **No Azure account needed.** These notebooks *simulate* Azure services in pure Python so you can learn the concepts without any cloud cost.

---
## 1. Azure Key Vault

Centralized store for three types of sensitive data:

| Type | Examples | Operations |
|------|---------|-------------|
| **Secrets** | API keys, connection strings, passwords | Get, Set, List, Delete |
| **Keys** | Encryption keys (RSA, EC) | Encrypt, Decrypt, Sign, Verify, Wrap, Unwrap |
| **Certificates** | TLS/SSL certificates | Create, Import, Renew |

### Key Vault access control

- **RBAC** (recommended): use Azure roles like `Key Vault Secrets User`.
- **Access policies** (legacy): per-principal permissions.
- **Network restrictions**: private endpoints, firewall rules.

### Bad practice → Best practice for secret handling

| ❌ Bad | 😐 Better | ✅ Best |
|-------|----------|--------|
| Hard-code secret in source code (`API_KEY = "sk-abc123"`) | Load from a local `.env` file | Fetch from Key Vault using a **Managed Identity** (no credentials on disk at all) |
| Anyone with repo access leaks the key | Dev machines still hold the plaintext | Rotation, audit logs, and RBAC all handled by Azure |

The cell below acts out that progression.

In [ ]:
# Simulate the bad → better → best journey for secrets handling
import os, textwrap

# ❌ BAD — hard-coded secret
def connect_to_db_bad():
    api_key = 'sk-live-ABC123-LEAKED-IN-GIT'  # 🚨 committed to source control!
    return f'Connecting with key={api_key[:10]}...'

# 😐 BETTER — .env file (still plaintext on disk, but out of git)
os.environ['API_KEY'] = 'sk-live-ABC123'  # pretend this was loaded from .env
def connect_to_db_better():
    api_key = os.environ['API_KEY']
    return f'Connecting with key={api_key[:10]}... (loaded from env)'

# ✅ BEST — Key Vault + Managed Identity, nothing sensitive in the app
class ManagedIdentity:
    def get_token(self, scope): return 'aad-token-xxx'

class KeyVaultClient:
    def __init__(self, vault_url, credential):
        self.vault_url = vault_url
        self.credential = credential
        self._secrets = {'api-key': 'sk-live-ABC123'}
    def get_secret(self, name):
        self.credential.get_token('https://vault.azure.net')  # auth transparently
        return self._secrets[name]

def connect_to_db_best():
    kv = KeyVaultClient('https://contoso-kv.vault.azure.net', ManagedIdentity())
    api_key = kv.get_secret('api-key')
    return f'Connecting with key={api_key[:10]}... (fetched from Key Vault via Managed Identity)'

print('❌ BAD   :', connect_to_db_bad())
print('😐 BETTER:', connect_to_db_better())
print('✅ BEST  :', connect_to_db_best())

### Simulating Key Vault operations

The next cell shows what really happens behind the scenes of a Key Vault: RBAC check, soft-delete, and recovery.

In [ ]:
# A mock Key Vault that mirrors Azure's RBAC + soft-delete behavior.

class MockKeyVault:
    READ_ROLES  = {'Key Vault Secrets User', 'Key Vault Administrator', 'Owner'}
    WRITE_ROLES = {'Key Vault Secrets Officer', 'Key Vault Administrator', 'Owner'}

    def __init__(self, soft_delete_days: int = 90, purge_protection: bool = True):
        self._secrets: dict[str, str]  = {}
        self._deleted: dict[str, str]  = {}
        self.soft_delete_days = soft_delete_days
        self.purge_protection = purge_protection

    def set_secret(self, name, value, role):
        if role not in self.WRITE_ROLES:
            print(f'  ❌ Access denied — role "{role}" cannot write secrets'); return
        self._secrets[name] = value
        print(f'  ✅ Secret "{name}" set (value hidden)')

    def get_secret(self, name, role):
        if role not in self.READ_ROLES:
            print(f'  ❌ Access denied — role "{role}" cannot read secrets'); return None
        val = self._secrets.get(name)
        print(f'  ✅ Secret "{name}" retrieved' if val else f'  ❌ Secret "{name}" not found')
        return val

    def delete_secret(self, name):
        if name in self._secrets:
            self._deleted[name] = self._secrets.pop(name)
            print(f'  🗑️  Secret "{name}" soft-deleted (recoverable for {self.soft_delete_days} days)')

    def recover_secret(self, name):
        if name in self._deleted:
            self._secrets[name] = self._deleted.pop(name)
            print(f'  ♻️  Secret "{name}" recovered from soft-delete!')

    def purge_secret(self, name):
        if self.purge_protection:
            print(f'  🛡️  Purge protection is ON — cannot permanently delete "{name}"')
            return
        self._deleted.pop(name, None)
        print(f'  💥 Secret "{name}" purged permanently')


vault = MockKeyVault()
print('=== Writing secrets ===')
vault.set_secret('db-conn', 'Server=prod.db;Password=s3cret', role='Key Vault Secrets Officer')
vault.set_secret('api-key', 'sk-abc123xyz',                   role='Key Vault Secrets Officer')

print('\n=== Reader tries to write (denied) ===')
vault.set_secret('foo', 'bar', role='Reader')

print('\n=== Authorized reader ===')
vault.get_secret('db-conn', role='Key Vault Secrets User')

print('\n=== Unauthorized reader ===')
vault.get_secret('db-conn', role='Reader')

print('\n=== Soft-delete + recover + purge ===')
vault.delete_secret('api-key')
vault.get_secret('api-key', role='Key Vault Secrets User')   # gone
vault.recover_secret('api-key')
vault.get_secret('api-key', role='Key Vault Secrets User')   # back!
vault.delete_secret('api-key')
vault.purge_secret('api-key')                                # blocked by purge protection

### Exam must-knows for Key Vault

- **Soft delete** is always on and cannot be turned off: deleted items stay recoverable for a configurable 7–90 days (default 90).
- **Purge protection** (optional) blocks permanent deletion during the soft-delete window — even by admins. Many compliance frameworks require it.
- **Tiers and FIPS validation** (a favourite exam distractor):

  | Tier | Key protection | FIPS validation |
  |---|---|---|
  | Key Vault **Standard** | Software-protected keys | FIPS 140 **Level 1** |
  | Key Vault **Premium** | HSM-protected keys in a multi-tenant HSM | FIPS 140-3 **Level 3** |
  | **Managed HSM** | HSM-protected keys in a *single-tenant*, fully isolated HSM pool | FIPS 140-3 **Level 3** |

  The jump from Standard to Premium is *software → HSM*. The jump from Premium to Managed HSM is *multi-tenant → single-tenant* (plus its own local RBAC model).
- Prefer **Managed Identity** over connection strings or SAS tokens.

---
## 2. Microsoft Defender for Cloud

Defender for Cloud is **CSPM** (Cloud Security Posture Management) and **CWPP** (Cloud Workload Protection Platform) bundled into one service.

### CSPM — "How secure am I?"

| Feature | What it does |
|---------|-------------|
| **Secure Score** | A 0-100% grade of your security posture. |
| **Recommendations** | Actionable fixes like *"Enable MFA for owner accounts"*. |
| **Security policies** | Built on Azure Policy; mapped to CIS, NIST, PCI DSS, etc. |
| **Regulatory compliance** | Dashboard showing compliance against industry standards. |
| **Cloud security graph** | A map of resources + attack paths (paid tier). |

### CWPP — "Protect my workloads"

| Defender plan | What it protects |
|--------------|-------------------|
| Defender for Servers | VMs — vulnerability scanning, endpoint detection |
| Defender for Storage | Blob/File storage — malware scanning |
| Defender for SQL | Azure SQL — vulnerability + anomaly detection |
| Defender for Containers | AKS — image scanning, runtime protection |
| Defender for App Service | Web apps — threat detection |
| Defender for Key Vault | Unusual secret access patterns |

> 🗓️ **Plan-naming currency note.** In 2026 Microsoft stopped taking *new* onboardings for a few standalone plans: **Defender for DNS** folded into Defender for Servers, **Defender for Key Vault** and **Defender for Resource Manager** moved to a fixed-price model, and **Defender for Kubernetes / Container Registry** were long since replaced by **Defender for Containers**. Existing customers keep the protection. Exam material still uses the plan names above, so learn them — just know that "enable the Defender for DNS plan" is no longer how you get DNS coverage.

### Free vs Paid

| | Free (Foundational CSPM) | Paid (Defender CSPM + plans) |
|-|--------|------|
| Secure score | ✅ | ✅ |
| Recommendations | Basic | Enhanced |
| Workload protection | ❌ | ✅ (per plan) |
| Attack path analysis | ❌ | ✅ |
| Agentless vulnerability scanning | ❌ | ✅ |

> **Exam tip**: the free tier gives you Secure Score and basic recommendations. Workload protection (CWPP) requires paid Defender plans.

In [ ]:
# Compute Secure Score and prioritize fixes by impact
RECOMMENDATIONS = [
    {'title': 'Enable MFA for accounts with owner permissions',  'max_score': 10, 'status': 'healthy'},
    {'title': 'Storage accounts should use private link',         'max_score': 8,  'status': 'unhealthy'},
    {'title': 'Enable disk encryption on VMs',                    'max_score': 6,  'status': 'healthy'},
    {'title': 'Enable Defender for SQL',                          'max_score': 4,  'status': 'unhealthy'},
    {'title': 'Restrict management ports with JIT',               'max_score': 8,  'status': 'unhealthy'},
    {'title': 'Enable Key Vault purge protection',                'max_score': 4,  'status': 'healthy'},
    {'title': 'Use RBAC for Key Vault access',                    'max_score': 6,  'status': 'healthy'},
    {'title': 'Enable Defender for Servers',                      'max_score': 4,  'status': 'not_applicable'},
]

applicable = [r for r in RECOMMENDATIONS if r['status'] != 'not_applicable']
max_total = sum(r['max_score'] for r in applicable)
current   = sum(r['max_score'] for r in applicable if r['status'] == 'healthy')
pct       = current / max_total * 100

print(f'=== Microsoft Defender for Cloud — Secure Score ===\n')
print(f'Score: {current}/{max_total} ({pct:.0f}%)\n')
print('Recommendations:')
icons = {'healthy': '✅', 'unhealthy': '🔴', 'not_applicable': '⬜'}
for r in RECOMMENDATIONS:
    impact = f'+{r["max_score"]} pts' if r['status'] == 'unhealthy' else ''
    print(f'  {icons[r["status"]]} {r["title"]:<55} {impact}')

# Prioritize: unhealthy items sorted by biggest score impact first
print('\n💡 Fix these first (highest impact):')
todo = sorted((r for r in RECOMMENDATIONS if r['status'] == 'unhealthy'),
              key=lambda r: -r['max_score'])
for r in todo:
    print(f'   {r["max_score"]:>2} pts → {r["title"]}')

### Just-in-Time (JIT) VM access — a CWPP superpower

Management ports like 22 (SSH) and 3389 (RDP) shouldn't be open 24/7. JIT **closes them by default** and opens them only when an authorized user requests temporary access.

The next cell simulates a JIT request: the port is closed → admin requests access → NSG opens for 1 hour → automatically reverts.

In [ ]:
from datetime import datetime, timedelta

class JITPolicy:
    def __init__(self, vm, port):
        self.vm = vm
        self.port = port
        self.open_until = None
        self.allowed_ip = None

    def status(self, now):
        if self.open_until and now < self.open_until:
            return f'🟢 OPEN to {self.allowed_ip} (until {self.open_until.strftime("%H:%M")})'
        return '🔒 CLOSED (default)'

    def request(self, ip, hours, now):
        self.allowed_ip = ip
        self.open_until = now + timedelta(hours=hours)
        print(f'[{now.strftime("%H:%M")}] ✅ JIT granted for {ip} on {self.vm}:{self.port} for {hours}h')

    def check_traffic(self, src_ip, now):
        allowed = (self.open_until and now < self.open_until and src_ip == self.allowed_ip)
        return '✅ ALLOW' if allowed else '🚫 DENY'

vm_jit = JITPolicy('prod-web-01', 22)
t = datetime(2024, 1, 1, 9, 0)

print('Before request:', vm_jit.status(t))
print('  attacker 203.0.113.5 → SSH :', vm_jit.check_traffic('203.0.113.5', t))

t = datetime(2024, 1, 1, 9, 30)
vm_jit.request(ip='198.51.100.42', hours=1, now=t)

print('\nDuring the JIT window (09:35):')
t = datetime(2024, 1, 1, 9, 35)
print('  admin 198.51.100.42 → SSH  :', vm_jit.check_traffic('198.51.100.42', t))
print('  attacker 203.0.113.5 → SSH :', vm_jit.check_traffic('203.0.113.5', t))

print('\n90 minutes later (11:00 — window expired):')
t = datetime(2024, 1, 1, 11, 0)
print('  admin 198.51.100.42 → SSH  :', vm_jit.check_traffic('198.51.100.42', t))
print('  status                     :', vm_jit.status(t))

### Other paid CWPP features worth knowing
- **File integrity monitoring (FIM)** — alerts when critical OS files or registry keys change. Now delivered through the Defender for Endpoint agent (it used to run on the retired Log Analytics/MMA agent).
- **Vulnerability assessment** — scans VMs and containers for CVEs, agent-based or **agentless**.
- **Agentless machine scanning** — snapshots the disk to find CVEs, secrets and malware without installing anything.
- **Attack path analysis / cloud security graph** — part of Defender CSPM, not of the workload plans.

> 🗓️ **Retired, do not learn these as current**: *adaptive application controls* and *adaptive network hardening* were deprecated in 2024 along with the Log Analytics (MMA) agent. Their jobs moved to Defender for Endpoint integration and agentless scanning. You may still see them in older SC-900 study guides.

---
## 3. Multi-cloud support

Defender for Cloud isn't Azure-only — remember these connectors for the exam:

- **AWS** via the **native AWS connector** — you connect an AWS account (or organization) directly; no AWS Security Hub required. Defender for Cloud *can* also push findings to Security Hub, which is where the older "Security Hub connector" wording came from.
- **GCP** via the **native GCP connector** (project or organization level).
- **On-premises** via **Azure Arc** (projects servers, Kubernetes and SQL into Azure as first-class resources so Policy and Defender can target them).

Per-server protection (Defender for Servers) on AWS/GCP/on-prem machines still needs **Azure Arc**; the native connectors alone give you agentless CSPM.

This gives you a **single Secure Score across all clouds**.

---
## Summary

| Concept | Key fact |
|---------|----------|
| **Key Vault** | Secrets, keys, certificates. Soft delete + purge protection + RBAC. |
| **Managed Identity** | Avoid putting credentials in code or .env files. |
| **Defender for Cloud** | CSPM + CWPP in one service. |
| **Secure Score** | 0-100% gauge, driven by recommendations. |
| **CSPM free tier** | Score + basic recommendations. |
| **CWPP (paid)** | Per-workload plans (Servers, SQL, Containers...). |
| **JIT VM access** | Ports closed by default, opened on demand for a time window. |
| **Multi-cloud** | AWS, GCP, and on-prem via Azure Arc. |

**Next**: [Notebook 3 — Sentinel and Defender XDR](03_sentinel_and_defender_xdr.ipynb)

---
## Self-check — Key Vault & Defender for Cloud

In [ ]:
QUIZ = [
    {'id': 'Q1',
     'q': 'An app running on an Azure VM needs a database password. Which design has no credential to leak?',
     'options': {'A': 'Password in an environment variable set by the deployment pipeline',
                 'B': 'Password in Key Vault, app authenticates with a Key Vault access key',
                 'C': 'Password in Key Vault, app authenticates with a managed identity',
                 'D': 'Password in an encrypted appsettings.json'},
     'a': 'C',
     'why': 'A managed identity is issued and rotated by Azure; there is no secret on disk, in the pipeline or in '
            'the repo. Every other option still moves a secret somewhere a human or a build log can see it.'},
    {'id': 'Q2',
     'q': 'You must guarantee that even a compromised Key Vault administrator cannot permanently destroy a key '
          'during the retention window. What do you enable?',
     'options': {'A': 'Soft delete', 'B': 'Purge protection', 'C': 'A resource lock', 'D': 'RBAC instead of access policies'},
     'a': 'B',
     'why': 'Soft delete is already always on and only makes deletion recoverable - an admin can still purge. '
            'Purge protection removes the purge operation for the whole retention window, for everyone.'},
    {'id': 'Q3',
     'q': 'Which capability is in the FREE tier of Defender for Cloud?',
     'options': {'A': 'Attack path analysis', 'B': 'Secure Score and security recommendations',
                 'C': 'Just-in-Time VM access', 'D': 'Agentless vulnerability scanning'},
     'a': 'B',
     'why': 'Foundational CSPM (free) gives Secure Score, recommendations and the MCSB compliance view. Attack '
            'paths and agentless scanning need Defender CSPM; JIT needs Defender for Servers P2.'},
    {'id': 'Q4',
     'q': 'A regulated workload needs a single-tenant, FIPS 140-3 Level 3 HSM that no other customer shares. Pick one.',
     'options': {'A': 'Key Vault Standard', 'B': 'Key Vault Premium', 'C': 'Azure Managed HSM', 'D': 'Azure Disk Encryption'},
     'a': 'C',
     'why': 'Premium is HSM-backed and also FIPS 140-3 Level 3, but the HSM pool is multi-tenant. Managed HSM gives '
            'you a dedicated, single-tenant pool. Standard is software-protected (FIPS 140 Level 1).'},
    {'id': 'Q5',
     'q': 'A hospital wants to protect 100 on-premises servers with Defender for Servers. What is the prerequisite?',
     'options': {'A': 'Nothing - Defender for Cloud discovers them automatically',
                 'B': 'Onboard them to Azure Arc', 'C': 'Move them to Azure', 'D': 'Enable the AWS connector'},
     'a': 'B',
     'why': 'Azure Arc projects a non-Azure machine into Azure Resource Manager so Azure Policy and Defender plans '
            'can target it. Without Arc there is no Azure resource to protect.'},
]

MY_ANSWERS = {'Q1': 'C', 'Q2': 'B', 'Q3': 'B', 'Q4': 'C', 'Q5': 'B'}

score = 0
for q in QUIZ:
    mine = MY_ANSWERS.get(q['id'], '').strip().upper()
    ok = mine == q['a']
    score += ok
    print(f'{"PASS" if ok else "FAIL"}  {q["id"]}: {q["q"]}')
    for k, v in q['options'].items():
        print(f'         {k}. {v} {"<-- correct" if k == q["a"] else ""}')
    print(f'         your answer: {mine or "(blank)"}')
    print(f'         why: {q["why"]}\n')
print(f'Score: {score}/{len(QUIZ)}')
